In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob

# Seperation 
 * distnace to nearest neighbour
 * relative angle to nearest neighbour

In [14]:
def distance_to_nearest_neighbour(no_nan_block):

    # checking theres no NaNs in the data
    if no_nan_block.isnull().values.any():
        raise ValueError(f'dataframe contains NaN values {no_nan_block[no_nan_block.isnull().any(axis=1)]}')
    
    # getting the x and y coordinates of fish 1
    x1 = no_nan_block.iloc[:, 0]
    y1 = no_nan_block.iloc[:, 1]

    distances = {}

    # calculating the distnace from fish 1 to each other fish at each time step
    for i in range(2, 6):
        # fish 2 to 5
        dx = no_nan_block.iloc[:, (i-1) * 3] - x1
        dy = no_nan_block.iloc[:, (i-1) * 3 + 1] - y1
        distances[i] = np.sqrt(dx**2 + dy**2)

    # finding the minimum distance across all other fish at each timestep
    distances_df = pd.DataFrame(distances)
    nearest_neighbour_distance = distances_df.min(axis=1)

    return nearest_neighbour_distance.iloc[1:].to_frame(name='nearest_neighbour_disttance')

In [15]:
def relative_angle_to_nearest_neighbour(no_nan_block):

    # checking theres no NaNs in the data
    if no_nan_block.isnull().values.any():
        raise ValueError(f'dataframe contains NaN values {no_nan_block[no_nan_block.isnull().any(axis=1)]}')
    
    # getting the x and y coordinates of fish 1
    x1 = no_nan_block.iloc[:, 0]
    y1 = no_nan_block.iloc[:, 1]
    heading1 = no_nan_block.iloc[:, 2]

    # calculating distance and angle from fish 1 to the other fish at each time step
    distances = {}
    angles = {}
    for i in range(2, 6):
        dx = no_nan_block.iloc[:, (i-1) * 3] - x1
        dy = no_nan_block.iloc[:, (i-1) * 3 + 1] - y1
        distances[i] = np.sqrt(dx**2 + dy**2)
        angles[i] = no_nan_block.iloc[:, (i-1) * 3 + 2] - heading1

    # finding the nearest neighbour at each time step
    distances_df = pd.DataFrame(distances)
    angles_df = pd.DataFrame(angles)
    nearest_neighbour_idx = distances_df.idxmin(axis=1)

    # looking up heading difference corresponding to the nearest neighbour
    relative_angle = pd.Series([angles_df.loc[t, nearest_neighbour_idx[t]] for t in distances_df.index], index=distances_df.index)

    return relative_angle.iloc[1:].to_frame(name='relative_angle_to_nearest_neighbour')

# Cohesion 
* distance to group centroid
* relative angle to group centroid

In [3]:
def centre_of_neighbours(positions_array):
    '''
    for each fish at each time step finding the mean
    position for all other visible (not nan) fish - the centre of the nighbours
    '''
    t, n, _ = positions_array.shape # t = timesteps, n = number of fish
    centres = np.full((t, n, 2), np.nan) # pre fill with NaN values so that if the centre cannot be calculated there will still be a NaN value in the array
    n_neighbours = np.zeros((t, n), dtype=int) # creating an empty count array to fill in the loop

    for f in range(n):
        # getting indicies of every fish expect fish f that is the one we are looping over
        otherfish_idx = [j for j in range(n) if j != f]
        otherfish = positions_array[:, otherfish_idx, :] # positions of all other fish

        # finding mean of all neihgbouring fish whilst ignoring the NaN values
        with np.errstate(all='ignore'): # suppressing warnings from all NaN slices
            centres[:, f, :] = np.nanmean(otherfish, axis=1)

        # counting how many neighbours are visible (not NaN) at each timestep
        visible_fish = ~np.isnan(otherfish[:, :, 0])
        n_neighbours[:, f] = visible_fish.sum(axis=1)

    return centres, n_neighbours


In [16]:
def distance_to_group_centroid(no_nan_block):

    # checking theres no NaNs in the data
    if no_nan_block.isnull().values.any():
        raise ValueError(f'dataframe contains NaN values {no_nan_block[no_nan_block.isnull().any(axis=1)]}')
    
    # getting x, y cooridinates for all 5 fish and stacking into an array
    positions = np.stack([no_nan_block.iloc[:, [(i-1)*3, (i-1)*3 + 1]].values for i in range(1, 6)], axis=1)

    # finding centroid - centre of neighbours
    centres, _ = centre_of_neighbours(positions)

    # getting the centroid of neighbours for fish 1 
    fish1_position = positions[:, 0, :]
    fish1_centroid = centres[:, 0, :]

    # calculating distance from fish 1 to the group centroid at each time step
    cohesion_vector = fish1_centroid - fish1_position
    distance = np.linalg.norm(cohesion_vector, axis=1)

    nearest_neighbour_distance = pd.Series(distance, index=no_nan_block.index)

    return nearest_neighbour_distance.iloc[1:].to_frame(name='distance_to_group_centroid')

In [17]:
def relative_angle_to_group_centroid(no_nan_block):

    # checking theres no NaNs in the data
    if no_nan_block.isnull().values.any():
        raise ValueError(f'dataframe contains NaN values {no_nan_block[no_nan_block.isnull().any(axis=1)]}')
    
    # getting x, y cooridinates for all 5 fish and stacking into an array
    positions = np.stack([no_nan_block.iloc[:, [(i-1)*3, (i-1)*3 + 1]].values for i in range(1, 6)], axis=1)

     # finding centroid - centre of neighbours
    centres, _ = centre_of_neighbours(positions)

    # getting fish 1s position, heading and centroid of its neighbours
    fish1_position = positions[:, 0, :]
    fish1_heading = no_nan_block.iloc[:, 2].values
    fish1_centroid = centres[:, 0, :]

    # calculating the angle of the vector from fish 1 to the centroid
    dx = fish1_centroid[:, 0] - fish1_position[:, 0]
    dy = fish1_centroid[:, 1] - fish1_position[:, 1]
    absolute_angle = np.arctan2(dy, dx)

    # subtracting fish 1s heading to get the relative angle
    relative_angle = absolute_angle - fish1_heading

    relative_angle_series = pd.Series(relative_angle, index=no_nan_block.index)

    return relative_angle_series.iloc[1:].to_frame(name='relative_angle_to_group_centroid')

# Alignment
* relative angle to the group heading

In [18]:
def relative_angle_to_group_heading(no_nan_block):

    # checking theres no NaNs in the data
    if no_nan_block.isnull().values.any():
        raise ValueError(f'dataframe contains NaN values {no_nan_block[no_nan_block.isnull().any(axis=1)]}')
    
    # gettign fish 1s heading 
    fish1_heading = no_nan_block.iloc[:, 2]

    # getting mean heading of all the other fish 
    other_headings = pd.DataFrame({i: no_nan_block.iloc[:, (i-1)*3 + 2] for i in range(2, 6)})
    mean_group_heading = other_headings.mean(axis=1)

    # finding the relative angle between fish 1s heading the mean of the group heading 
    relative_angle = mean_group_heading - fish1_heading

    relative_angle_series = pd.Series(relative_angle, index=no_nan_block.index)
    
    return relative_angle_series.iloc[1:].to_frame(name='relative_angle_to_group_heading')

# Movement 
* velocity
* angular velocity

In [19]:
def finding_scalar_speed(no_nan_block):

    # checking theres no NaNs in the data
    if no_nan_block.isnull().values.any():
        raise ValueError(f'dataframe contains NaN values {no_nan_block[no_nan_block.isnull().any(axis=1)]}')
    
    # getting fish 1s x y coordinates
    x1 = no_nan_block.iloc[:, 0]
    y1 = no_nan_block.iloc[:, 1]

    # claculating the displacement between consecutive timesteps
    dx = x1.diff()
    dy = y1.diff()

    # calculating scalar speed (magnitude of displacement)
    speed = np.sqrt(dx**2 + dy**2)

    return speed.iloc[1:].to_frame(name='scalar_speed_fish1')


In [20]:
def finding_velocity(no_nan_block):
    
    # checking theres no NaNs in the data
    if no_nan_block.isnull().values.any():
        raise ValueError(f'dataframe contains NaN values {no_nan_block[no_nan_block.isnull().any(axis=1)]}')
    
    # getting fish 1s x y coordinates
    x1 = no_nan_block.iloc[:, 0]
    y1 = no_nan_block.iloc[:, 1]

    # calculating the displacement between consecutive timesteps
    dx = x1.diff()
    dy = y1.diff()

    # 2 column data frame
    return pd.DataFrame({'vx_fish1': dx, 'vy_fish1': dy}, index=no_nan_block.index).iloc[1:]

2 functions for velocity - not sure which you want

In [21]:
def finding_angular_velocity(no_nan_block):

    # checking theres no NaNs in the data
    if no_nan_block.isnull().values.any():
        raise ValueError(f'dataframe contains NaN values {no_nan_block[no_nan_block.isnull().any(axis=1)]}')
    
    # getting fish 1s heading
    heading1 = no_nan_block.iloc[:, 2]

    # claculating the change in heading between consecutive time steps
    angular_velocity = heading1.diff()

    angular_velocity_series = pd.Series(angular_velocity, index=no_nan_block.index)

    return angular_velocity_series.iloc[1:].to_frame(name='angular_velocity_fish1')

# Position
* distance to the nearest wall
* angle to centre

In [22]:
def distance_to_nearest_wall(no_nan_block):

    # checking theres no NaNs in the data
    if no_nan_block.isnull().values.any():
        raise ValueError(f'dataframe contains NaN values {no_nan_block[no_nan_block.isnull().any(axis=1)]}')
    
    # converting wall radius fromcm to mm to match fish coordinates
    inner_radius = 25 * 10
    outer_radius = 35 * 10

    # getting fish 1s x y coordinates
    x1 = no_nan_block.iloc[:, 0]
    y1 = no_nan_block.iloc[:, 1]

    # calculating distnace from fish 1 to centre (0, 0)
    distance_to_centre = np.sqrt(x1**2 + y1**2)

    # claculating distance to each wall
    distance_to_inner_wall = np.abs(distance_to_centre - inner_radius)
    distance_to_outer_wall = np.abs(distance_to_centre - outer_radius)

    # findinf the nearest wall
    nearest_wall_distance = np.minimum(distance_to_inner_wall, distance_to_outer_wall)

    nearest_wall_distance_series = pd.Series(nearest_wall_distance, index=no_nan_block.index)
    return nearest_wall_distance_series.iloc[1:].to_frame(name='distance_to_nearest_wall')

In [23]:
def angle_to_centre(no_nan_block):

    # checking theres no NaNs in the data
    if no_nan_block.isnull().values.any():
        raise ValueError(f'dataframe contains NaN values {no_nan_block[no_nan_block.isnull().any(axis=1)]}')
    
    # getting fish 1s x y coordinates and heading
    x1 = no_nan_block.iloc[:, 0]
    y1 = no_nan_block.iloc[:, 1]
    heading1 = no_nan_block.iloc[:, 2]

    # calculating the angle of vector from fish 1 to centre (0, 0)
    dx = 0 - x1
    dy = 0 - y1
    absolute_angle = np.arctan2(dy, dx)

    # subtracting fish 1a heading to get relative angle
    relative_angle = absolute_angle - heading1

    relative_angle_series = pd.Series(relative_angle, index=no_nan_block.index)

    return relative_angle_series.iloc[1:].to_frame(name='relative_angle_to_centre')